# NB-04: Self-Attention and Cross-Attention

## Learning Objectives
- Understand the SelfAttention module with RoPE on Q/K and the asymmetry that V is NOT normalized
- Trace the CrossAttention dual-stream path (CLIP image + T5 text) with the hardcoded split at index 257
- Verify the hardcoded CLIP split at index 257 (1 CLS + 256 patch tokens from ViT-H/14)
- Compare parameter counts between self-attention and cross-attention variants

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm), NB-02 (QKV projections and multi-head layout), NB-03 (3D RoPE -- applied to Q/K in self-attention)
- **Assumed concepts:** Attention mechanism, dot-product similarity, softmax

## Concept Map
- `SelfAttention` -> used in DiTBlock (NB-05, NB-06 Track 2)
- `CrossAttention` -> used in DiTBlock for conditioning (NB-06 Track 2)
- `flash_attention` -> backend dispatch function (NB-02 explained layouts)

> The VACE encoder reuses this same attention mechanism with an additional control signal path -- covered in Track 4.

In [1]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios (up to 6 levels up).
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import the symbols this notebook covers
from diffsynth.models.wan_video_dit import (
    SelfAttention, CrossAttention, precompute_freqs_cis_3d,
    rope_apply, flash_attention, RMSNorm, AttentionModule
)
import torch
import torch.nn as nn
from einops import rearrange

print("Setup complete.")

Project root: /home/basheer/Signapse/Codes/wan22-architecture-course
Setup complete.


## 1. SelfAttention Architecture

**WHY:** Self-attention lets each video token attend to every other token, building global context across the entire spatial-temporal grid. This is the core mechanism that allows the DiT to model long-range dependencies in video.

**HOW:** The `SelfAttention` module combines four linear projections (q, k, v, o), two RMSNorm instances (norm_q, norm_k), and one AttentionModule:

1. **Four linear projections (q, k, v, o):** All `dim -> dim`. No dimension expansion -- head splitting happens inside the attention backend via einops rearrange.
2. **norm_q and norm_k -- RMSNorm applied BEFORE RoPE:** Both `q` and `k` are passed through their respective `RMSNorm` instances before RoPE is applied. This stabilizes the attention dot-product by bounding the magnitude of the query and key vectors.
3. **v is NOT normalized:** The value projection `v = self.v(x)` goes directly into attention without any normalization. Values carry the actual content to be mixed; normalizing them would change the output magnitude.
4. **RoPE applied after normalization:** `rope_apply(q, freqs, num_heads)` rotates Q and K in the complex plane using precomputed 3D frequencies.

Source: `diffsynth/models/wan_video_dit.py`, line 125

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 125
dim, num_heads = 384, 4  # reduced config
head_dim = dim // num_heads  # 96

sa = SelfAttention(dim=dim, num_heads=num_heads)
print("SelfAttention components:")
for name, module in sa.named_children():
    if hasattr(module, 'weight'):
        print(f"  {name}: {type(module).__name__}, weight {module.weight.shape}")
    else:
        print(f"  {name}: {type(module).__name__}")
total_params = sum(p.numel() for p in sa.parameters())
assert head_dim == 96, f"Expected head_dim=96, got {head_dim}"
assert sa.q.weight.shape == torch.Size([dim, dim]), f"Q weight shape mismatch"
print(f"\nTotal parameters: {total_params:,}")

## 2. SelfAttention Forward Pass

The forward pass steps for `SelfAttention.forward(x, freqs)`:

```
x  [B, S, dim]
  |-- q = q_proj(x)      [B, S, dim]  (linear projection)
  |   q = norm_q(q)      [B, S, dim]  (RMSNorm: stabilizes dot-product magnitude)
  |   q = rope_apply(q)  [B, S, dim]  (RoPE: encodes 3D position into q)
  |
  |-- k = k_proj(x)      [B, S, dim]  (linear projection)
  |   k = norm_k(k)      [B, S, dim]  (RMSNorm: same reason as q)
  |   k = rope_apply(k)  [B, S, dim]  (RoPE: encodes 3D position into k)
  |
  |-- v = v_proj(x)      [B, S, dim]  (linear projection -- NO normalization)
  |
  attn(q, k, v)           [B, S, dim]  (flash_attention SDPA fallback on CPU)
  o = o_proj(attn_out)    [B, S, dim]  (output projection)
```

Note that RoPE is applied AFTER normalization (line 142-146 in the source). The normalization brings Q/K to unit variance first, then RoPE rotates the resulting vectors.

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 141
B = 1
F_grid, H_grid, W_grid = 4, 4, 4
S = F_grid * H_grid * W_grid  # 64 video tokens

# Build 3D frequency grid (same as NB-03)
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(dim // num_heads)
freqs = torch.cat([
    f_freqs[:F_grid].view(F_grid, 1, 1, -1).expand(F_grid, H_grid, W_grid, -1),
    h_freqs[:H_grid].view(1, H_grid, 1, -1).expand(F_grid, H_grid, W_grid, -1),
    w_freqs[:W_grid].view(1, 1, W_grid, -1).expand(F_grid, H_grid, W_grid, -1),
], dim=-1).reshape(S, 1, -1)

x = torch.randn(B, S, dim)
with torch.no_grad():
    out = sa(x, freqs)
assert out.shape == torch.Size([B, S, dim]), f"Expected {(B, S, dim)}, got {out.shape}"
print(f"Input:  {x.shape}")
print(f"Freqs:  {freqs.shape}")
print(f"Output: {out.shape}")

## 3. Q/K Normalization in Context

The RMSNorm on Q and K affects the attention score distribution. The norms bring Q and K to unit variance, which stabilizes the dot product scores. Without normalization, the variance of Q and K depends on the linear projection weights, which can vary widely during training.

In [ ]:
with torch.no_grad():
    q_raw = sa.q(x)
    q_normed = sa.norm_q(q_raw)
    k_raw = sa.k(x)
    k_normed = sa.norm_k(k_raw)
    v_raw = sa.v(x)
assert q_raw.shape == torch.Size([B, S, dim]), f"Q raw shape mismatch"
assert q_normed.shape == torch.Size([B, S, dim]), f"Q normed shape mismatch"
print(f"Q raw std: {q_raw.std():.4f}  ->  Q normed std: {q_normed.std():.4f}")
print(f"K raw std: {k_raw.std():.4f}  ->  K normed std: {k_normed.std():.4f}")
print(f"V raw std: {v_raw.std():.4f}  (no normalization)")

## 4. CrossAttention: Dual-Stream Conditioning

**WHY:** CrossAttention handles conditioning from two sources simultaneously. Video tokens (queries) attend to external context (keys/values) rather than to each other.

**HOW:** When `has_image_input=True`, the context `y` is a concatenated tensor from two modalities:

1. **CLIP image embeddings:** 257 tokens (1 CLS + 256 patch tokens from ViT-H/14). Hardcoded split at index 257.
2. **T5 text embeddings:** Variable-length text tokens.

**Why separate streams:** Image and text conditions have different semantic structures -- CLIP provides visual context, T5 provides textual instructions. They use separate K/V projections but share the same Q (from video tokens). The outputs are summed (additive fusion).

The forward pass runs two separate attention operations:
1. Text cross-attention: `attn(q, norm_k(k(ctx)), v(ctx))` -- video tokens attend to T5 text
2. Image cross-attention: `flash_attention(q, norm_k_img(k_img(img)), v_img(img))` -- same query, different K/V from CLIP
3. Results are summed before the output projection: `x = x + y`

> The VACE encoder reuses this same attention mechanism with an additional control signal path -- covered in Track 4.

Source: `diffsynth/models/wan_video_dit.py`, line 151

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 151
B, S_vid, S_text, dim, num_heads = 1, 64, 30, 384, 4

ca = CrossAttention(dim=dim, num_heads=num_heads, has_image_input=True)
x = torch.randn(B, S_vid, dim)          # video tokens (queries)
y = torch.randn(B, 257 + S_text, dim)   # CLIP(257) + T5 text(30) = 287 tokens
# Inside forward: img = y[:, :257], ctx = y[:, 257:]

with torch.no_grad():
    out = ca(x, y)
assert out.shape == torch.Size([B, S_vid, dim]), f"Expected {(B, S_vid, dim)}, got {out.shape}"
print(f"Video tokens (Q):     {x.shape}")
print(f"Context (CLIP + T5):  {y.shape}  (257 CLIP + {S_text} T5 = {257 + S_text})")
print(f"Output:               {out.shape}")
print(f"\nCLIP split: y[:, :257]  -> img [{B}, 257, {dim}]")
print(f"T5 split:   y[:, 257:]  -> ctx [{B}, {S_text}, {dim}]")

## 5. CrossAttention Without Image Input

When `has_image_input=False`, the entire context `y` is treated as text-only. No image branch, no split at 257. This mode is used when only text conditioning is available.

In [ ]:
ca_text = CrossAttention(dim=dim, num_heads=num_heads, has_image_input=False)
y_text = torch.randn(B, S_text, dim)  # text only, no CLIP
with torch.no_grad():
    out_text = ca_text(x, y_text)
assert out_text.shape == torch.Size([B, S_vid, dim])
print(f"Text-only context: {y_text.shape}")
print(f"Output:            {out_text.shape}")

# Verify: no image parameters when has_image_input=False
img_params = [n for n, _ in ca_text.named_parameters() if 'img' in n]
assert len(img_params) == 0, "has_image_input=False should have no image parameters"
print(f"\nImage-related parameters (should be empty): {img_params}")

## 6. Parameter Count Comparison

Comparing SelfAttention vs CrossAttention (with and without image input) reveals the cost of the dual-stream architecture. The image branch adds `k_img`, `v_img` (each `dim x dim` weight + `dim` bias), and `norm_k_img` (`dim` weight).

In [ ]:
sa_params = sum(p.numel() for p in sa.parameters())
ca_params = sum(p.numel() for p in ca.parameters())
ca_text_params = sum(p.numel() for p in ca_text.parameters())
# Image branch should add k_img + v_img + norm_k_img parameters
assert ca_params > ca_text_params, "Image branch should add parameters"
assert ca_params > sa_params, "CrossAttention with image should have more params than SelfAttention"
print(f"SelfAttention:                    {sa_params:>10,} params")
print(f"CrossAttention (with image):      {ca_params:>10,} params")
print(f"CrossAttention (text only):       {ca_text_params:>10,} params")
print(f"\nImage branch adds: {ca_params - ca_text_params:,} params (k_img, v_img, norm_k_img)")

## Summary

### Key Takeaways
- **SelfAttention with RoPE:** Q and K are each passed through RMSNorm before RoPE is applied, but V is NOT normalized. This bounds the dot-product magnitude while preserving value content scale.
- **CrossAttention dual-stream:** When `has_image_input=True`, context `y` is split at index 257: `img = y[:, :257]` (1 CLIP CLS + 256 patches) and `ctx = y[:, 257:]` (T5 text). Two attention operations run with the same Q but different K/V; results are summed.
- **257 CLIP split:** The constant 257 is hardcoded and comes from CLIP ViT-H/14: 1 CLS token + 256 patch tokens = 257.
- **Q/K normalization:** RMSNorm on Q and K brings them to unit variance, stabilizing attention scores regardless of projection weight magnitude.

### Source References

| Symbol | Location |
|--------|----------|
| `SelfAttention` | `diffsynth/models/wan_video_dit.py`, line 125 |
| `CrossAttention` | `diffsynth/models/wan_video_dit.py`, line 151 |
| `flash_attention` | `diffsynth/models/wan_video_dit.py`, line 28 |
| `AttentionModule` | `diffsynth/models/wan_video_dit.py`, line 115 |

## Exercises

### Exercise 1 -- Attention score magnitude
Extract the raw attention weights by modifying `flash_attention` to use `F.scaled_dot_product_attention` with `attn_mask=None` and check the magnitude of `Q @ K^T`. How does Q/K normalization affect these scores? Compare the score magnitude before and after RMSNorm.

### Exercise 2 -- Different context lengths
Change `S_text` to 50 and 100. Verify CrossAttention output shape remains `[B, S_vid, dim]` regardless of context length. Why does the output shape not depend on the context length?

### Exercise 3 -- Additive vs concatenative fusion
The current implementation sums the text and image attention outputs (`x = x + y` at line 186). What would change if they were concatenated instead? Consider the impact on output dimension and downstream compatibility.